# 9.9 CatBoost

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 9: Ensemble Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-09-ensemble-learning/09.09_catboost.ipynb)

Companion code for Section 9.9 of the book. Run the cells in order from top to bottom.

**Example 9.9.1**

In [ ]:
from catboost import CatBoostClassifier, Pool, cv as catboost_cv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import LabelEncoder
import warnings; warnings.filterwarnings('ignore')

# ── CatBoost on numerical data ────────────────────────────────────
data = load_breast_cancer()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

cat_model = CatBoostClassifier(
    iterations      = 500,
    learning_rate   = 0.05,
    depth           = 6,
    l2_leaf_reg     = 3.0,
    border_count    = 128,   # Histogram bins
    eval_metric     = 'AUC',
    early_stopping_rounds = 50,
    random_seed     = 42,
    verbose         = 0
)
cat_model.fit(X_tr, y_tr, eval_set=(X_te, y_te))

print(f'CatBoost Best iteration: {cat_model.best_iteration_}')
print(f'CatBoost Test Accuracy:  {cat_model.score(X_te, y_te):.4f}')
print(f'CatBoost Test AUC:       {roc_auc_score(y_te, cat_model.predict_proba(X_te)[:,1]):.4f}')

# ── CatBoost with native categorical features ─────────────────────
np.random.seed(42)
n = 2000
df_cat = pd.DataFrame({
    'age':            np.random.randint(18, 70, n),
    'income':         np.random.exponential(60000, n),
    'education':      np.random.choice(['HS','BS','MS','PhD'], n),
    'job_type':       np.random.choice(['tech','finance','healthcare','retail','education'], n),
    'city_tier':      np.random.choice(['Tier1','Tier2','Tier3'], n),
    'device':         np.random.choice(['mobile','desktop','tablet'], n),
    'country':        np.random.choice(['US','UK','DE','IN','AU'], n),
    'subscription':   np.random.choice(['free','basic','premium'], n),
})

# Create target with some signal
df_cat['churn'] = ((df_cat['age'] > 50) | (df_cat['subscription']=='free')).astype(int)
df_cat['churn'] = df_cat['churn'] ^ np.random.choice([0,0,0,1], n)  # Add noise

cat_features = ['education','job_type','city_tier','device','country','subscription']
X_cat = df_cat.drop('churn', axis=1)
y_cat = df_cat['churn']

X_cat_tr, X_cat_te, y_cat_tr, y_cat_te = train_test_split(
    X_cat, y_cat, test_size=0.2, stratify=y_cat, random_state=42)

# Method 1: CatBoost with native categorical handling
cat_native = CatBoostClassifier(
    iterations=200, learning_rate=0.1, depth=4,
    cat_features=cat_features,   # Just specify which columns are categorical!
    random_seed=42, verbose=0
)
cat_native.fit(X_cat_tr, y_cat_tr, eval_set=(X_cat_te, y_cat_te))
acc_native = cat_native.score(X_cat_te, y_cat_te)
auc_native = roc_auc_score(y_cat_te, cat_native.predict_proba(X_cat_te)[:,1])
print(f'\nCatBoost (native categoricals): acc={acc_native:.4f}  AUC={auc_native:.4f}')

# Method 2: Manual one-hot encoding + XGBoost
import xgboost as xgb
X_cat_ohe = pd.get_dummies(X_cat, columns=cat_features)
X_cat_ohe_tr = X_cat_ohe.loc[X_cat_tr.index]
X_cat_ohe_te = X_cat_ohe.loc[X_cat_te.index]
xgb_ohe = xgb.XGBClassifier(n_estimators=200, use_label_encoder=False,
                              eval_metric='logloss', random_state=42, verbosity=0)
xgb_ohe.fit(X_cat_ohe_tr, y_cat_tr)
acc_ohe = xgb_ohe.score(X_cat_ohe_te, y_cat_te)
auc_ohe = roc_auc_score(y_cat_te, xgb_ohe.predict_proba(X_cat_ohe_te)[:,1])
print(f'XGBoost (one-hot encoded):      acc={acc_ohe:.4f}  AUC={auc_ohe:.4f}')
print(f'\nCatBoost wins with {X_cat.shape[1]} cols vs {X_cat_ohe.shape[1]} OHE cols - no encoding needed!')

# ── Feature importance ────────────────────────────────────────────
fi = cat_native.get_feature_importance(prettified=True)
print('\nCatBoost Feature Importance (categorical + numerical):')
print(fi.head(10))

# ── Visualise ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('CatBoost Analysis', fontsize=14, fontweight='bold')

ax = axes[0]
fi_plot = fi.head(8)
ax.barh(fi_plot['Feature Id'], fi_plot['Importances'], color='#9C27B0', edgecolor='white')
ax.set_xlabel('Importance'); ax.set_title('CatBoost Feature Importance\n(including categorical features)')
ax.grid(alpha=0.3, axis='x')

ax = axes[1]
methods = ['CatBoost\n(native cats)','XGBoost\n(one-hot)']
accs    = [acc_native, acc_ohe]
aucs    = [auc_native, auc_ohe]
x = np.arange(2)
w = 0.3
ax.bar(x-w/2, accs, w, label='Accuracy', color='#1976D2', edgecolor='white')
ax.bar(x+w/2, aucs, w, label='AUC',      color='#43A047', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(methods)
ax.set_ylim(0.60, 0.95); ax.set_ylabel('Score')
ax.set_title('CatBoost vs XGBoost\n(Mixed categorical/numerical data)')
ax.legend(); ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('catboost_analysis.png', dpi=150, bbox_inches='tight')
plt.show()